Q: "What does 1536-dimensional embedding mean?"
An embedding is a dense vector representation of text. A 1536-dimensional embedding means the model converts any input text into a vector containing 1536 floating-point numbers. These numbers collectively encode the semantic meaning of the text, allowing similar texts to have vectors that are close together in vector space.

Vectorless RAG

Instead of converting text into embeddings, we organize the document into a hierarchical tree.

PDF
   ↓
Parse
   ↓
Sections
   ↓
Subsections
   ↓
Paragraphs
   ↓
Sentences

That itself becomes a tree.

Example PDF:

Insurance Guide

1. Auto Insurance
      Premium
      Claims
      Deductibles

2. Home Insurance
      Fire
      Theft
      Flood

Tree:

                    Insurance Guide
                     /           \
            Auto Insurance    Home Insurance
             /      |      \      /   |   \
      Premium Claims Deductibles Fire Theft Flood

Every node stores text.

What is inside a node?

A node might contain

Node

title:
    Premium

content:
    Premium is the amount paid every month...

children:
    []

parent:
    Auto Insurance

or

{
 title,
 text,
 children,
 parent
}
How is the tree built?

Suppose PDF contains

Chapter 1

Car Insurance

Premium

Premium depends on age.

Claims

Claim process...

Chapter 2

Health Insurance

Coverage

...

Exclusions

...

Parser identifies headings.

Chapter 1

↓

Child

Car Insurance

↓

Children

Premium

Claims

↓

Paragraphs

Premium depends on age...

Tree becomes

Chapter 1
      |
Car Insurance
   /       \
Premium   Claims
   |          |
paragraph   paragraph
Why make a tree?

Because documents already have hierarchy.

A sentence belongs to

Sentence
↓

Paragraph
↓

Section
↓

Chapter
↓

Document

That relationship is valuable.

Embeddings ignore this structure.

Retrieval

Suppose user asks

How is premium calculated?

Instead of

Embedding(question)

↓

Vector Search

we do

Question

↓

Find matching title

↓

Go to Premium node

↓

Read its children

↓

Return answer
Another example

Question

How do I file a claim?

Search starts at top.

Insurance Guide

↓

Look at children

Auto Insurance
Health Insurance

"Claim" probably belongs to Auto.

↓

Open Auto node

Premium
Claims
Deductible

↓

Go to Claims

↓

Read paragraph

To file a claim...

Done.

But how do we find the node?

Good question.

There are several approaches.

1. Keyword matching
Question

↓

claim

↓

Find nodes containing "claim"
2. BM25

Instead of vectors,

Question

↓

BM25

↓

Top nodes

Very common.

3. Tree traversal with an LLM

LLM is shown only the children.

Example

Root

Insurance Guide

Children:

Auto Insurance
Home Insurance
Travel Insurance

Question

How do deductibles work?

LLM chooses

Auto Insurance

Now only its children are shown.

Premium
Claims
Deductible

LLM chooses

Deductible

Now only that paragraph is loaded.

This is sometimes called top-down hierarchical retrieval.

Why is this called Vectorless?

Because nowhere did we compute

Sentence

↓

Embedding

↓

1536 numbers

Instead we use

Document structure

+
keywords

+
BM25

+
LLM reasoning

No vector database is required.

Building the tree from a PDF

Pipeline:

                PDF
                 |
         Extract text
                 |
       Detect headings
                 |
     Detect hierarchy levels
                 |
        Create tree nodes
                 |
      Attach paragraphs
                 |
          Store tree

Example final tree:

Document
│
├── Chapter 1
│      │
│      ├── Premium
│      │      ├── paragraph
│      │      └── paragraph
│      │
│      └── Claims
│             ├── paragraph
│             └── paragraph
│
└── Chapter 2
       │
       ├── Coverage
       └── Exclusions
Is this always built from PDF headings?

Not necessarily.

Well-structured PDFs: Use headings, font sizes, numbering (e.g., 1, 1.1, 2.3) to build the hierarchy.
Poorly structured PDFs: Use layout analysis and heuristics (font size, bold text, spacing), or even an LLM to infer sections.
Scanned PDFs: Run OCR first, then infer the hierarchy.
Vector RAG vs Vectorless Tree RAG
Vector RAG	Vectorless Tree RAG
Split into chunks	Build a document hierarchy
Create embeddings	Keep headings and parent-child relationships
Store in vector DB	Store as a tree
Similarity search	Tree traversal + BM25/keywords/LLM
Fast semantic search	Preserves document structure and context

Step 1: PDF aaya

Maan lo PDF hai:

Insurance Guide

1. Auto Insurance

Premium

Premium depends on age and driving history.

Claims

To file a claim...

2. Home Insurance

Coverage

Fire damage...

Abhi ye sirf text hai.

Step 2: Parser PDF ko read karta hai

LLM nahi.

Usually libraries use hoti hain:

pdfplumber
PyMuPDF (fitz)
Unstructured
Docling
Azure Document Intelligence
Google Document AI

Ye sirf text hi nahi nikalti, balki ye bhi batati hain:

Text:
Premium

Font Size:
20

Bold:
Yes

Position:
x=100
y=320

Aur paragraph ke liye:

Premium depends on age...

Font Size:
11

Bold:
No

Matlab parser ko formatting bhi pata chalti hai.

Step 3: Heading identify hoti hai

Ab parser dekhta hai

Insurance Guide

Font = 32

↓

Root node.

Fir dekha

Auto Insurance

Font = 24

↓

Child node.

Fir

Premium

Font = 18

↓

Aur ek child.

Fir

Premium depends on age...

Font = 11

↓

Ye heading nahi hai.

Ye Premium node ka content ban jayega.

Tree:

Insurance Guide
      |
Auto Insurance
      |
Premium
      |
Premium depends on age...
Node banta kaise hai?

Programming mein literally object banta hai.

Example:

class Node:
    def __init__(self, title):
        self.title = title
        self.content = ""
        self.children = []

Jab parser "Premium" dekhta hai

premium = Node("Premium")

Ho gaya node create.

Fir

premium.content = "Premium depends on age..."

Fir

auto.children.append(premium)

Ho gaya tree.

Isme koi LLM nahi.

LLM kab use hota hai?
Case 1

Acha PDF hai.

Heading

Subheading

Paragraph

LLM ki zarurat hi nahi.

Parser enough hai.

Case 2

Bakwas PDF hai.

Jaise

AUTO INSURANCE

Premium depends on age

Claim Process

Upload documents

Deductible

Yaha parser confuse ho sakta hai.

Tab prompt diya ja sakta hai:

These are extracted blocks.

Identify hierarchy.

Return JSON.

LLM return karega

{
 "Auto Insurance": {
   "Premium": "...",
   "Claim Process": "...",
   "Deductible": "..."
 }
}

Ab us JSON se tree ban jayega.

To haan, kuch systems mein LLM tree banata hai, lekin har system mein nahi.

Search hamesha BM25 hota hai?

Bilkul nahi.

Bahut options hain.

Method 1

Keyword Search

Question

How to file claim?

↓

Search

claim

↓

Node mil gaya.

Method 2

BM25

Question

↓

BM25 score

↓

Top 5 nodes.

Ye Elasticsearch, Lucene, OpenSearch mein common hai.

Method 3

LLM Tree Traversal

Sabse interesting.

Question:

How is deductible calculated?

Root

Insurance Guide

Children

Auto Insurance
Home Insurance
Travel Insurance

LLM se poocha

Question:

How is deductible calculated?

Choose one child.

LLM

Auto Insurance

Fir sirf uske children.

Premium
Claims
Deductible

Fir

Deductible

Fir paragraph.

Ye ek decision tree ki tarah chal raha hai.

Method 4

Hybrid

Pehle BM25.

Top 10 nodes.

Fir LLM choose karega.

Ye bahut common hai.

Question
      |
BM25
      |
Top 10 Nodes
      |
LLM
      |
Best Node
Method 5

Rule Based

Question

claim

Agar

claim

to

Claims Section

Agar

premium

to

Premium Section

Simple mapping.

Real companies kya karti hain?

Depends.

Small PDFs

Parser only.

PDF

↓

Parser

↓

Tree
Enterprise

Parser

↓

LLM fixes hierarchy

↓

Tree

↓

BM25

↓

LLM

↓

Answer

Very advanced systems

Parser

↓

Knowledge Graph

↓

Tree

↓

BM25

↓

Cross Encoder

↓

LLM

Ye kaafi sophisticated setup hota hai.

Ek important baat

Tree banana aur search karna do alag stages hain.

Indexing time (sirf ek baar)
PDF
   ↓
Parser
   ↓
Identify headings
   ↓
Create Node objects
   ↓
Connect parent-child
   ↓
Save Tree

Ye tree ek baar ban gaya.

Query time (har question par)
Question
   ↓
Keyword / BM25 / LLM
   ↓
Relevant Node
   ↓
Read children/parent if needed
   ↓
Context
   ↓
LLM Answer

To har query pe tree dobara nahi banta. Tree pehle se stored hota hai, aur query aane par usi tree mein traversal hoti hai.


Isliye poora tree kabhi LLM ko nahi diya jata. Sirf jis node par ho, uske children fetch kiye jaate hain.

Flow kuch aisa hota hai:

Your PDF
    │
    ▼
PageIndex API
    │
    ├── Parse PDF
    ├── Build hierarchical tree
    ├── (Optional) Generate node summaries
    ├── Store the tree
    └── Return a document ID


User Question
      │
      ▼
Your Backend
      │
      ▼
PageIndex Query API
      │
      ▼
PageIndex traverses the tree
      │
      ▼
Relevant pages/nodes
      │
      ▼
LLM generates the answer

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client=OpenAI(api_key=OPENAI_API_KEY)


In [ ]:
PDF_PATH='./SystemDesignInterview.pdf'
print(f"Uploading {PDF_PATH} to PageIndex...")
result=pi_client.upload_file(PDF_PATH)
doc_id=result['doc_id']

Server ke paas

doc_1
Insurance.pdf

doc_2
FastAPI.pdf

doc_3
SystemDesign.pdf

Ab agar baad me question poochna ho:

Explain Kafka partitions.

To server ko kaise pata chalega kis PDF se answer dena hai?

In [ ]:
while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result['status']
    if status=='ready':
        print(f"Document {doc_id} is ready for querying.")
        break
    elif status=='failed':
        print(f"Document {doc_id} failed to process.")
        break
    
    time.sleep(5)

In [ ]:
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get('result',[])